## 데이터 분석 기초

### 회귀와 분류
**(1) 회귀**
- 연속적인 숫자 값을 예측하는 문제
- 집값 예측, 기온 예측, 주식 가격 예측

**(2) 분류**
- 클래스(범주)를 예측하는 문제
- 이메일이 스팸인지 아닌지, 질병진단(정상/고혈압/당뇨), 사진 속 동물 분류
<br><br>
> 2 유형은 회귀, 이진분류, 다중분류 중 하나의 유형이 시험에 출제

### 데이터 분석 프로세스
1) 탐색적 데이터 분석
2) 데이터 전처리
3) 데이터 셋 분할
4) 학습 
5) 평가
6) 예측 및 결과 저장

=> 4,5 - 회귀, 분류 모델에 따라 코드 달라지며, 나머지는 공통사항 <br>
=> 1,2,3,6 - 어떤 분석이든 똑같이 진행

### 1) 탐색적 데이터 분석
#### 1-1) 문제파악
- 제공된 학습용 데이터(elec_train.csv)는 전국 건물의 기상 정보(기온, 강수량, 풍속, 습도) 및 전력 소비량을 기록한 자료이다. <br>
    해당 데이터를 기반으로 전력 소비량을 예측하는 회귀 모델을 개발하고, 가장 우수한 모델을 평가 데이터(elec_test.csv)에 적용하여 <br>
    전력 소비량을 예측하시오. 예측 결과는 아래의 [제출 형식]을 준수하여, CSV 파일로 생성하는 코드를 제출하시오.

> [제출형식] CSV 파일명: result.csv, 전력 소비량 컬럼명: pred (1개)

In [3]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('../sample_data/part2/회귀/전기사용량/elec_train.csv')
test = pd.read_csv('../sample_data/part2/회귀/전기사용량/elec_test.csv')

# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

#### 1-2) 데이터 유형 파악
- 각 변수의 데이터 유형(숫자형, 범주형 등)을 파악해 이후 전처리 방향을 결정

In [4]:
# 데이터 유형 파악
print(train.info())
print(test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5100 entries, 0 to 5099
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   건물코드    5100 non-null   object 
 1   기온      5100 non-null   float64
 2   강수량     1114 non-null   float64
 3   풍속      5099 non-null   float64
 4   습도      5099 non-null   float64
 5   전력소비량   5100 non-null   float64
dtypes: float64(5), object(1)
memory usage: 239.2+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630 entries, 0 to 629
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   건물코드    630 non-null    object 
 1   기온      630 non-null    float64
 2   강수량     630 non-null    float64
 3   풍속      630 non-null    float64
 4   습도      630 non-null    int64  
dtypes: float64(3), int64(1), object(1)
memory usage: 24.7+ KB
None


#### 1-3) 결측치 파악
- 결측치는 모델 성능 저하 원인이 되므로 사전에 파악하여 적절히 처리 필요

In [5]:
# 결측치 파악
print(train.isnull().sum())
print(test.isnull().sum())

건물코드        0
기온          0
강수량      3986
풍속          1
습도          1
전력소비량       0
dtype: int64
건물코드    0
기온      0
강수량     0
풍속      0
습도      0
dtype: int64


#### 1-4) 범주형 변수 카테고리 탐색
- 범주형 변수는 학습 전 카테고리 수와 구성 확인이 필요

In [6]:
# 범주형 변수 카테고리 파악
print(train['건물코드'].value_counts()) # 빈도수 계산하는 함수
print(test['건물코드'].value_counts())

건물코드
CM    51
DP    51
BE    51
AH    51
DH    51
      ..
BM    51
BR    51
AQ    51
AG    51
DL    51
Name: count, Length: 100, dtype: int64
건물코드
AH    7
CY    7
CB    7
DF    7
CG    7
     ..
CP    6
BL    6
CM    6
CC    6
AC    6
Name: count, Length: 100, dtype: int64


### 2) 데이터 전처리
#### 2-1) X, Y 데이터 셋 분리
- 독립변수(X)와 종속변수(y)를 분리하는 작업으로, 모델 학습 시 독립변수만 활용

In [7]:
# X, Y 데이터 셋 분리
X_train = train.drop(['전력소비량'], axis=1) # 행(0) 열(1)
y = train['전력소비량']

print(X_train.shape, y.shape, test.shape)

(5100, 5) (5100,) (630, 5)


#### 2-2) 결측치 처리
- 결측치는 평균값, 최빈갓 대체 또는 제거 방식 등으로 처리하여 학습 오류를 방지
> 제거 방식은 비추천함

**결측값 채우기**
- 특정 값으로 채우기: `df['컬럼값'].fillna(0)`
- 열의 평균값으로 채우기: `df['컬럼값'].fillna(df['컬럼값'].mean())`
- 열의 최빈값으로 채우기: `df['컬럼값'].fillna(df['컬럼값'].mode()[0])`

In [8]:
# 결측치 처리
X_train['강수량'] = X_train['강수량'].fillna(0)
X_train['풍속'] = X_train['풍속'].fillna(X_train['풍속'].mean())
X_train['습도'] = X_train['습도'].fillna(X_train['풍속'].mode()[0])

print(X_train.isnull().sum())

건물코드    0
기온      0
강수량     0
풍속      0
습도      0
dtype: int64


#### 2-3) 수치형 변수 스케일링
- 모델이 특정 변수에 치우치지 않도록 값을 일정 범위로 조정
    - MinMaxScaling: 0~1 사이로 정규화
    - StandardScaling: 평균 0, 표준편차 1로 정규화

In [23]:
# 수치형 변수 스케일링 - MinMaxScaling
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# 수치형 변수들만 추출
num_columns = X_train.select_dtypes(exclude='object').columns #select_dtypes: 특정 데이터 타입만 가져오는 함수

X_train[num_columns] = scaler.fit_transform(X_train[num_columns]) # 학습에서는 fit(학습) - transform(변형) 을 같이 해줘야함.
test[num_columns] = scaler.transform(test[num_columns]) # 학습에서 만든 fit_transform 결과 값을 담은 scaler를 그대로 적용하기 위해 transform(변형) 적용

print(X_train.head())

  건물코드        기온       강수량        풍속        습도
0   CM  0.791837  0.000000  0.308271  0.836401
1   DR  0.612245  0.000000  0.203008  0.989775
2   CY  0.355102  0.000000  0.180451  1.000000
3   BS  0.469388  0.007752  0.135338  0.948875
4   BN  0.861224  0.000000  0.082707  0.621677


#### 2-4) 범주형 변수 인코딩
- 모델이 이해할 수 있도록 문자 데이터를 숫자로 변환
    - LabelEncoding: 순서형에 적합 
    - OneHotEncoding: 명목형에 적합 (중복 피하기 위해 다중 컬럼화)
    - ex) 예시
        | 범주형 변수 | Label | OneHot | 
        | -------- | ------ | ------ |
        | A | 0 | [0,0,1] |
        | B | 1 | [0,1,0] |
        | C | 2 | [1,0,0] |

    


In [24]:
# 범주형 변수 인코딩 - LabelEncoding
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

X_train['건물코드'] = encoder.fit_transform(X_train['건물코드'])
test['건물코드'] = encoder.transform(test['건물코드'])
print(X_train['건물코드'])

0       64
1       95
2       76
3       44
4       39
        ..
5095    72
5096    14
5097    61
5098    19
5099    86
Name: 건물코드, Length: 5100, dtype: int64


### 3. 학습용, 검증용 데이터 셋 분할
- 과적합 방지를 위해 학습용/검증용 데이터로 분리하여 일반화 성능을 평가

In [25]:
# 학습, 검증 데이터 셋 분할
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2) #train데이터, 종속변수 넣어주기, 학습용 데이터 80%, 검증용데이터 20%로 분할
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(4080, 5) (1020, 5) (4080,) (1020,)


### 4. Randomforest 활용 모델 학습
- 학습용 데이터를 기반으로 모델을 학습시키는 과정으로, 분류 또는 회귀 문제에 따라 모델이 달라짐.

In [26]:
# Randomforest 활용 모델 학습
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### 5. MSE, R2 Score 활용 평가
- 예측 결과와 실제값의 차이를 수치로 나타내 모델 성능을 확인
    - 회귀: MSE, RMSE, R2 등
    - 분류: F1-Score, ROC_AUC 등

In [27]:
# MSE, R2 Score 활용 평가
from sklearn.metrics import mean_squared_error, r2_score
y_pred = model.predict(X_val) # train: 학습 -> validation: 검증
mse = mean_squared_error(y_val, y_pred) #검증값과 예측값의 차이
r2 = r2_score(y_val, y_pred) #0~1 사이값 나옴, 1에 가까울수록 성능이 좋음.
print(mse, r2)

646337.5048121003 0.8784779838361382


### 6. test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
- 최종 모델을 test 데이터에 적용해 예측값을 생성하고, 제출 양식에 맞게 저장

In [32]:
# test 데이터 예측 및 결과 저장(pred 1개 컬럼)
test_pred = model.predict(test)
result = pd.DataFrame(test_pred, columns=['pred'])
result.to_csv('11_result.csv', index=False)